In [9]:
input_file = '/home/ajarrah/PhD_Thesis/chapter_3/aggregated_h5ad_data/aggregated_mouse_brain_202502.h5ad'
input_folder = "/home/ajarrah/PhD_Thesis/chapter_3/h5ad_data/"
output_folder = "/home/ajarrah/PhD_Thesis/chapter_4/h5ad_data/genes_all"


In [10]:
# Core scverse libraries
import scanpy as sc
import numpy as np
import pandas as pd
import os
import numpy as np

# Data retrieval
import pooch

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor="white")

/tmp/ipykernel_166238/1954675120.py:12: FutureWarning: Use `scanpy.set_figure_params` instead
  sc.settings.set_figure_params(dpi=80, facecolor="white")


In [11]:
adata = sc.read_h5ad(input_file)

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [12]:
aad_1 = sc.read_h5ad(os.path.join(input_folder, "A1_Aged_AD_Mouse_Brain_202502.h5ad"))
aad_2 = sc.read_h5ad(os.path.join(input_folder, "B1_Aged_AD_Mouse_Brain_202502.h5ad"))
aad_3 = sc.read_h5ad(os.path.join(input_folder, "C1_Aged_AD_Mouse_Brain_202502.h5ad"))
aad_4 = sc.read_h5ad(os.path.join(input_folder, "D1_Aged_AD_Mouse_Brain_202502.h5ad"))
ac_1 = sc.read_h5ad(os.path.join(input_folder, "A1_Aged_Control_Mouse_Brain_202502.h5ad"))
ac_2 = sc.read_h5ad(os.path.join(input_folder, "B1_Aged_Control_Mouse_Brain_202502.h5ad"))
ac_3 = sc.read_h5ad(os.path.join(input_folder, "C1_Aged_Control_Mouse_Brain_202502.h5ad"))
ac_4 = sc.read_h5ad(os.path.join(input_folder, "D1_Aged_Control_Mouse_Brain_202502.h5ad"))
yad_1 = sc.read_h5ad(os.path.join(input_folder, "A1_Young_AD_Mouse_Brain_202502.h5ad"))
yad_2 = sc.read_h5ad(os.path.join(input_folder, "B1_Young_AD_Mouse_Brain_202502.h5ad"))
yad_3 = sc.read_h5ad(os.path.join(input_folder, "C1_Young_AD_Mouse_Brain_202502.h5ad"))
yad_4 = sc.read_h5ad(os.path.join(input_folder, "D1_Young_AD_Mouse_Brain_202502.h5ad"))
yc_1 = sc.read_h5ad(os.path.join(input_folder, "A1_Young_Control_Mouse_Brain_202502.h5ad"))
yc_2 = sc.read_h5ad(os.path.join(input_folder, "B1_Young_Control_Mouse_Brain_202502.h5ad"))
yc_3 = sc.read_h5ad(os.path.join(input_folder, "C1_Young_Control_Mouse_Brain_202502.h5ad"))
yc_4 = sc.read_h5ad(os.path.join(input_folder, "D1_Young_Control_Mouse_Brain_202502.h5ad"))

In [13]:
data_name = [yc_1, yc_2, yc_3, yc_4,
                yad_1, yad_2, yad_3, yad_4,
                ac_1, ac_2, ac_3, ac_4,
                aad_1, aad_2, aad_3, aad_4]

sample_name = ["YC_1", "YC_2", "YC_3", "YC_4",
                "YAD_1", "YAD_2", "YAD_3", "YAD_4",
                "AC_1", "AC_2", "AC_3", "AC_4",
                "AAD_1", "AAD_2", "AAD_3", "AAD_4"]

sample_display_name = ["Young Control 1", "Young Control 2", "Young Control 3", "Young Control 4",
                       "Young AD 1", "Young AD 2", "Young AD 3", "Young AD 4",
                       "Aged Control 1", "Aged Control 2", "Aged Control 3", "Aged Control 4",
                       "Aged AD 1", "Aged AD 2", "Aged AD 3", "Aged AD 4"]

In [14]:

def add_coordinate_in_um(adata, spot_spacing_um=100.0):

    # Vectorized coordinate calculation
    adata.obs["y_um"] = adata.obs["array_row"] * (np.sqrt(3)/2 * spot_spacing_um)
    adata.obs["x_um"] = adata.obs["array_col"] * (spot_spacing_um / 2)

    # Round to 2 decimal places
    adata.obs["x_um"] = adata.obs["x_um"].round(2)
    adata.obs["y_um"] = adata.obs["y_um"].round(2)

    # Return only x/y with barcode as index
    return adata

In [15]:
# Ensure matrix is dense or use .A1 for sparse matrices
if isinstance(adata.X, np.ndarray):
    non_zero_genes = adata.X.sum(axis=0) != 0
else:
    non_zero_genes = np.array((adata.X.sum(axis=0) != 0)).ravel()  # for sparse matrix

adata = adata[:, non_zero_genes].copy()

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/anndata.py:1823: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [16]:
# Create output directory if it does not exist
os.makedirs(output_folder, exist_ok=True)

for i, adata in enumerate(data_name):

    adata_processed = add_coordinate_in_um(adata, spot_spacing_um=100.0).copy()

    adata_processed.write_h5ad(f"{output_folder}/{sample_name[i]}.h5ad")


In [17]:
data_name[0].obs

,in_tissue,array_row,array_col,Original_Name,Sample_Code,Group,y_um,x_um
AAACACCAATAACTGC-1,1,59,19,A1_Young_Control_Mouse_Brain_202502,1-1,YC,5109.55,950.0
AAACAGGGTCTATATT-1,1,47,13,A1_Young_Control_Mouse_Brain_202502,1-1,YC,4070.32,650.0
AAACAGTGTTCCTGGG-1,1,73,43,A1_Young_Control_Mouse_Brain_202502,1-1,YC,6321.99,2150.0
AAACCGGGTAGGTACC-1,1,42,28,A1_Young_Control_Mouse_Brain_202502,1-1,YC,3637.31,1400.0
AAACCGTTCGTCCAGG-1,1,52,42,A1_Young_Control_Mouse_Brain_202502,1-1,YC,4503.33,2100.0
...,...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,A1_Young_Control_Mouse_Brain_202502,1-1,YC,2684.68,3850.0
TTGTTTCACATCCAGG-1,1,58,42,A1_Young_Control_Mouse_Brain_202502,1-1,YC,5022.95,2100.0
TTGTTTCATTAGTCTA-1,1,60,30,A1_Young_Control_Mouse_Brain_202502,1-1,YC,5196.15,1500.0
TTGTTTCCATACAACT-1,1,45,27,A1_Young_Control_Mouse_Brain_202502,1-1,YC,3897.11,1350.0
